# 0.0 Imports

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import sklearn.metrics as mt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline

# 0.1 - Load Datasets

In [5]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks\regressao
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [6]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [7]:
# Instanciar o modelo com parâmetros default (grau 1)
model_def = Pipeline([
    ('features', PolynomialFeatures(degree=1)),
    ('model', ElasticNet())
])

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [8]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

--- Performance no Treino (Default) ---
R²:   0.0078
MSE:  474.27
RMSE: 21.78
MAE:  17.30
MAPE: 873.23%


## Passo 3 — Performance na validação (default)

In [9]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

--- Performance na Validação (Default) ---
R²:   0.0081
MSE:  473.64
RMSE: 21.76
MAE:  17.26
MAPE: 869.40%


## Passo 4 — Ajuste de hiperparâmetros

In [10]:
print("--- Testando Regressão Polinomial (Grau 2) com Elastic Net ---")
melhor_alpha_en = 1.0
melhor_l1_en = 0.5
melhor_r2_en = -999

# Parâmetros baixos e enxutos para evitar loops infinitos ou underfitting imediato
list_alpha_en = [0.001, 0.01, 0.1]
list_l1_ratio = [0.2, 0.5, 0.8]

for alpha in list_alpha_en:
    for l1_rat in list_l1_ratio:
        # Cria o pipeline composto para o Elastic Net
        model_poly_en = Pipeline([
            ('features_polinomiais', PolynomialFeatures(degree=2, include_bias=False)),
            ('regressor_elastic', ElasticNet(alpha=alpha, l1_ratio=l1_rat, max_iter=2000, random_state=42))
        ])
        
        try:
            model_poly_en.fit(X_train, y_train.values.ravel())
            yhat_train = model_poly_en.predict(X_train)
            yhat_val = model_poly_en.predict(X_val)
            
            r2_train = mt.r2_score(y_train, yhat_train)
            r2_val = mt.r2_score(y_val, yhat_val)
            rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
            
            print(f"Grau: 2 | Alpha: {alpha:<5} | L1_Ratio: {l1_rat:.1f} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f}")
            
            if r2_val > melhor_r2_en:
                melhor_r2_en = r2_val
                melhor_alpha_en = alpha
                melhor_l1_en = l1_rat
                
        except Exception as e:
            print(f"Erro no Elastic Net Alpha {alpha} | L1 {l1_rat}: {e}")

print("=" * 85)
print(f"-> VENCEDOR DO ENSAIO POLINOMIAL + ELASTIC NET:")
print(f"Melhor Alpha: {melhor_alpha_en}")
print(f"Melhor L1_Ratio: {melhor_l1_en}")
print(f"Maior R² obtido na Validação: {melhor_r2_en:.4f}\n")

--- Testando Regressão Polinomial (Grau 2) com Elastic Net ---
Grau: 2 | Alpha: 0.001 | L1_Ratio: 0.2 | R² Treino: 0.0898 | R² Val: 0.0674 | RMSE Val: 21.10
Grau: 2 | Alpha: 0.001 | L1_Ratio: 0.5 | R² Treino: 0.0907 | R² Val: 0.0676 | RMSE Val: 21.10


c:\Users\Guilherme Grandim\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.586e+03, tolerance: 5.042e+02
  model = cd_fast.enet_coordinate_descent(


Grau: 2 | Alpha: 0.001 | L1_Ratio: 0.8 | R² Treino: 0.0920 | R² Val: 0.0679 | RMSE Val: 21.10
Grau: 2 | Alpha: 0.01  | L1_Ratio: 0.2 | R² Treino: 0.0796 | R² Val: 0.0639 | RMSE Val: 21.14
Grau: 2 | Alpha: 0.01  | L1_Ratio: 0.5 | R² Treino: 0.0809 | R² Val: 0.0648 | RMSE Val: 21.13
Grau: 2 | Alpha: 0.01  | L1_Ratio: 0.8 | R² Treino: 0.0833 | R² Val: 0.0662 | RMSE Val: 21.12
Grau: 2 | Alpha: 0.1   | L1_Ratio: 0.2 | R² Treino: 0.0602 | R² Val: 0.0519 | RMSE Val: 21.28
Grau: 2 | Alpha: 0.1   | L1_Ratio: 0.5 | R² Treino: 0.0611 | R² Val: 0.0535 | RMSE Val: 21.26
Grau: 2 | Alpha: 0.1   | L1_Ratio: 0.8 | R² Treino: 0.0640 | R² Val: 0.0561 | RMSE Val: 21.23
-> VENCEDOR DO ENSAIO POLINOMIAL + ELASTIC NET:
Melhor Alpha: 0.001
Melhor L1_Ratio: 0.8
Maior R² obtido na Validação: 0.0679



## Passo 5 — Performance com modelo tunado (treino e validação)

In [11]:
# Modelo com os melhores hiperparâmetros encontrados, treinado apenas em X_train
model_tunado = Pipeline([
    ('features_polinomiais', PolynomialFeatures(degree=2, include_bias=False)),
    ('regressor_elastic', ElasticNet(alpha=melhor_alpha_en, l1_ratio=melhor_l1_en, max_iter=2000, random_state=42))
])
model_tunado.fit(X_train, y_train.values.ravel())

yhat_train_tunado = model_tunado.predict(X_train)
yhat_val_tunado   = model_tunado.predict(X_val)

r2_train_tunado   = mt.r2_score(y_train, yhat_train_tunado)
mse_train_tunado  = mt.mean_squared_error(y_train, yhat_train_tunado)
rmse_train_tunado = np.sqrt(mse_train_tunado)
mae_train_tunado  = mt.mean_absolute_error(y_train, yhat_train_tunado)
mape_train_tunado = mt.mean_absolute_percentage_error(y_train, yhat_train_tunado)

r2_val_tunado   = mt.r2_score(y_val, yhat_val_tunado)
mse_val_tunado  = mt.mean_squared_error(y_val, yhat_val_tunado)
rmse_val_tunado = np.sqrt(mse_val_tunado)
mae_val_tunado  = mt.mean_absolute_error(y_val, yhat_val_tunado)
mape_val_tunado = mt.mean_absolute_percentage_error(y_val, yhat_val_tunado)

print("--- Performance com Modelo Tunado ---")
print(f"Treino    → R²: {r2_train_tunado:.4f} | RMSE: {rmse_train_tunado:.2f} | MAE: {mae_train_tunado:.2f} | MAPE: {mape_train_tunado*100:.2f}%")
print(f"Validação → R²: {r2_val_tunado:.4f} | RMSE: {rmse_val_tunado:.2f} | MAE: {mae_val_tunado:.2f} | MAPE: {mape_val_tunado*100:.2f}%")

--- Performance com Modelo Tunado ---
Treino    → R²: 0.0920 | RMSE: 20.83 | MAE: 16.48 | MAPE: 838.91%
Validação → R²: 0.0679 | RMSE: 21.10 | MAE: 16.74 | MAPE: 857.83%


c:\Users\Guilherme Grandim\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.586e+03, tolerance: 5.042e+02
  model = cd_fast.enet_coordinate_descent(


## Passo 6 — União treino + validação

In [12]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

X_train_final shape: (15068, 13)
y_train_final shape: (15068, 1)


## Passo 7 — Retreino com melhores parâmetros

In [13]:
# Retreino com os melhores hiperparâmetros sobre o conjunto final (treino + validação)
model_final = Pipeline([
    ('features_polinomiais', PolynomialFeatures(degree=2, include_bias=False)),
    ('regressor_elastic', ElasticNet(alpha=melhor_alpha_en, l1_ratio=melhor_l1_en, max_iter=2000, random_state=42))
])
model_final.fit(X_train_final, y_train_final.values.ravel())

,steps,"[('features_polinomiais', ...), ('regressor_elastic', ...)]"
,transform_input,None
,memory,None
,verbose,False
,degree,2
,interaction_only,False
,include_bias,False
,order,'C'
,alpha,0.001
,l1_ratio,0.8
,fit_intercept,True


## Passo 8 — Performance no teste

In [14]:
# Predição no conjunto de teste (única vez que X_test é tocado)
yhat_test = model_final.predict(X_test)

# Métricas no conjunto de TESTE
r2_test   = mt.r2_score(y_test, yhat_test)
mse_test  = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

print("--- Performance no Teste (Final) ---")
print(f"R²:   {r2_test:.4f}")
print(f"MSE:  {mse_test:.2f}")
print(f"RMSE: {rmse_test:.2f}")
print(f"MAE:  {mae_test:.2f}")
print(f"MAPE: {mape_test * 100:.2f}%")

--- Performance no Teste (Final) ---
R²:   0.0886
MSE:  443.76
RMSE: 21.07
MAE:  16.76
MAPE: 833.11%


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [15]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   r2_train_tunado,   r2_val_tunado,   r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, rmse_train_tunado, rmse_val_tunado, rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  mae_train_tunado,  mae_val_tunado,  mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%', f'{mape_val_def*100:.2f}%',
              f'{mape_train_tunado*100:.2f}%', f'{mape_val_tunado*100:.2f}%',
              f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))


--- Quadro Comparativo — Diagnóstico Completo ---
           Conjunto       R²      RMSE       MAE    MAPE
   Treino (Default) 0.007832 21.777715 17.299507 873.23%
Validação (Default) 0.008117 21.763171 17.262903 869.40%
    Treino (Tunado) 0.091994 20.833580 16.484840 838.91%
 Validação (Tunado) 0.067860 21.097572 16.738029 857.83%
      Teste (Final) 0.088601 21.065642 16.758880 833.11%
